# Maia Fine-tuning Notebook (Gemma 4 E4B → Maia)

**Free Colab T4 GPU** | **Unsloth** | **QLoRA 4-bit** | **128K Context**

## Pasos
1. Runtime → Change runtime type → T4 GPU (gratis)
2. Run All
3. Pegar HF_TOKEN cuando lo pida (debe haber aceptado licencia Gemma 4)
4. Esperar ~10-12h
5. Modelo final se sube a HuggingFace Hub automáticamente

## 1. Setup

In [ ]:
import os
# === CONFIGURACIÓN ===
HF_TOKEN = os.environ.get('HF_TOKEN', '')  # se inyecta desde Colab Secrets
if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass('HuggingFace token (read+write): ')
os.environ['HF_TOKEN'] = HF_TOKEN

BASE_MODEL = 'unsloth/gemma-4-E4B-it'  # versión Unsloth (más rápida, 4-bit pre-quantized)
OUTPUT_REPO = 'maia-gemma4-e4b'  # nombre del repo final en HF
MAX_SEQ_LEN = 8192  # durante training (Gemma 4 E4B soporta 128K nativos en inferencia)
INFERENCE_CTX = 131072  # 128K para inferencia final
EPOCHS = 1
LR = 2e-4
BATCH_SIZE = 2
GRAD_ACCUM = 4

In [ ]:
# Instalar Unsloth (más rápido que HF puro en T4)
!pip install -q -U unsloth
!pip install -q -U datasets trl peft accelerate bitsandbytes huggingface_hub
!pip install -q sentencepiece protobuf

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=True)
print('HF login OK')

## 2. Cargar dataset desde el repo Maia

In [ ]:
!git clone --depth 1 https://github.com/calitosaa/Maia /content/maia_repo
%cd /content/maia_repo/finetuning/output
# Reassemble split parts
!cat maia_gemma4_finetune.jsonl.part_* > maia_gemma4_finetune.jsonl
!wc -l maia_gemma4_finetune.jsonl

In [ ]:
import json
from datasets import Dataset

DATASET_PATH = '/content/maia_repo/finetuning/output/maia_gemma4_finetune.jsonl'
examples = []
with open(DATASET_PATH) as f:
    for line in f:
        try:
            ex = json.loads(line)
            if 'messages' in ex and len(ex['messages']) >= 2:
                examples.append({'messages': ex['messages']})
        except Exception:
            continue
ds = Dataset.from_list(examples)
print(f'Dataset: {len(ds)} examples')
print(ds[0])

## 3. Cargar modelo Gemma 4 E4B con Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,            # auto: float16 en T4
    load_in_4bit=True,     # QLoRA 4-bit
    token=HF_TOKEN,
)

# Aumentar context window del modelo a 128K para inferencia final
model.config.max_position_embeddings = INFERENCE_CTX
tokenizer.model_max_length = INFERENCE_CTX
print(f'Model loaded. Training ctx={MAX_SEQ_LEN}, inference ctx={INFERENCE_CTX}')

In [ ]:
# Aplicar LoRA con configuración optimizada para T4
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',  # CRITICAL para extender contexto y reducir VRAM
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()

## 4. Format dataset con Gemma chat template

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template='gemma')

def format_examples(examples):
    texts = []
    for msgs in examples['messages']:
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {'text': texts}

ds = ds.map(format_examples, batched=True, remove_columns=['messages'])
print(ds[0]['text'][:500])

## 5. Train con TRL SFTTrainer + checkpoints a Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/maia_checkpoints'
!mkdir -p $OUTPUT_DIR

In [ ]:
from trl import SFTTrainer, SFTConfig

config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    logging_steps=20,
    save_steps=500,
    save_total_limit=3,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    fp16=True,
    optim='adamw_8bit',
    weight_decay=0.01,
    seed=42,
    report_to='none',
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    args=config,
)

trainer_stats = trainer.train()
print(trainer_stats)

## 6. Save merged model + push a HuggingFace Hub

In [ ]:
# Guardar adapter LoRA
model.save_pretrained(f'{OUTPUT_DIR}/maia-lora')
tokenizer.save_pretrained(f'{OUTPUT_DIR}/maia-lora')

# Merge + save final (16-bit)
model.save_pretrained_merged(f'{OUTPUT_DIR}/maia-final', tokenizer, save_method='merged_16bit')

# Push to HF Hub
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
USERNAME = api.whoami()['name']
REPO_ID = f'{USERNAME}/{OUTPUT_REPO}'
model.push_to_hub_merged(REPO_ID, tokenizer, save_method='merged_16bit', token=HF_TOKEN)
print(f'Pushed: https://huggingface.co/{REPO_ID}')

## 7. Convert to GGUF (Ollama / llama.cpp)

In [ ]:
# Save GGUF Q4_K_M (~2.5GB) y push
model.save_pretrained_gguf(f'{OUTPUT_DIR}/maia-gguf', tokenizer, quantization_method='q4_k_m')
model.push_to_hub_gguf(f'{REPO_ID}-GGUF', tokenizer, quantization_method='q4_k_m', token=HF_TOKEN)
print(f'GGUF pushed: https://huggingface.co/{REPO_ID}-GGUF')

## 8. Test inference

Para usar el modelo final:
```bash
ollama pull <USERNAME>/maia-gemma4-e4b-GGUF
ollama run maia
```

In [ ]:
FastLanguageModel.for_inference(model)
messages = [{'role': 'user', 'content': '¿Quién eres y qué puedes hacer?'}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])